In [ ]:
from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.utils import FRED
from SymbolicDSGE.utils.math_utils import HP_two_sided
import sympy as sp
from warnings import catch_warnings, simplefilter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
conf, kalman = ModelParser("../MODELS/POST82.yaml").get_all()

In [ ]:
with catch_warnings():
    simplefilter(action="ignore")
    mat = sp.Matrix(conf.equations.model)
mat

In [ ]:
sol = DSGESolver(conf, kalman)
comp = sol.compile(variable_order=conf.variables.variables, n_state=3, n_exog=2)
print(conf.variables)

In [ ]:
solved = sol.solve(
    comp,
    steady_state=np.asarray([0.0, 0.0, 0.0, 0.0, 0.0], dtype=float),
)

In [ ]:
solved.policy.eig

In [ ]:
params = {
    p.name: float(conf.calibration.parameters[p])
    for p in conf.parameters
    if p in conf.calibration.parameters
}

# state at time t
s = np.array([0.05, 0.077, 0.06])  # or any test state
P = solved.policy.p
F = solved.policy.f

# controls at time t (jump variables)
c = F @ s

cur = np.concatenate([s, c])

# expected next state (NO shock)
s1 = P @ s
c1 = F @ s1
fwd = np.concatenate([s1, c1])

res = solved.compiled.equations(fwd, cur, params)
print(res)

In [ ]:
solved.transition_plot(25, ["g", "z"], 1.0, observables=True)

In [ ]:
T = 200
g_shock = Shock(T, "norm", seed=0, dist_kwargs={"loc": 0.0}).shock_generator()
z_shock = Shock(T, "norm", seed=1, dist_kwargs={"loc": 0.0}).shock_generator()

sim_shocks = {"g": g_shock, "z": z_shock}

# sim_shocks = np.array([[1.0, 1.0], *np.zeros((24, 2))])
sol = solved.sim(T, sim_shocks, observables=True)

In [ ]:
sol_plot = sol.copy()
del sol_plot["_X"]
n_plots = len(sol_plot)
dim = np.ceil(np.sqrt(n_plots)).astype(int)
fig, ax = plt.subplots(dim, dim, figsize=(15, 10))
ax = ax.flatten()
while len(ax) > n_plots:
    fig.delaxes(ax[-1])
    ax = ax[:-1]

for i, (var, series) in enumerate(sol_plot.items()):
    ax[i].plot(series)
    ax[i].set_title(var)
    ax[i].grid(linestyle=":")
plt.tight_layout()
plt.show()

In [ ]:
# fred test
f = FRED(key_name="FRED_KEY")
df = f.get_frame(
    series_ids=["GDPC1", "CPIAUCSL", "FEDFUNDS"],
    date_range=("1960-01-01", "1997-10-01"),
)

In [ ]:
time_idx = pd.date_range(start="1960-01-01", end="1997-10-01", freq="QS")
df = df.reindex(time_idx)
df

In [ ]:
# Convert to model units
df["GDPC1"] = 100 * (np.log(df["GDPC1"]) - HP_two_sided(np.log(df["GDPC1"]), 1600)[0])
df["CPIAUCSL"] = 400 * np.log((df["CPIAUCSL"] / df["CPIAUCSL"].shift(1)).dropna())

In [ ]:
df = df.loc[df.index >= "1982-01-01"]

In [ ]:
df.GDPC1.plot()

In [ ]:
x0 = [0.0, 0.0, df["FEDFUNDS"].iloc[0], df["CPIAUCSL"].iloc[0], df["GDPC1"].iloc[0]]
solved.sim(T=df.shape[0], shocks=None, observables=True, x0=x0)